---
title: Fine-Tuning Large Language Models 
date: November 2025
authors: <i>COMET Team</i></br>; prAxIs team; Irene Berezin
description: 'An introduction to fine-tuning LLMs using economic articles, in Python.'
categories:
  - python
  - fine-tuning
  - sentiment-analysis
  - LLMs
  - AI
format:
  html: default
  ipynb:
    jupyter:
      kernelspec:
        display_name: Python
        language: python3
        name: python3
---

***
This notebook provides an introduction to fine-tuning a large-language model (LLM) for the purpose of analyzing the content and composition of thousands of economic articles. 

## 0. Prerequisites 

### 0.1 Prior Knowledge

- A basic understanding of coding in Python. 
### 0.2 Hardware/Software requirements

- Conda/miniconda installed on your device.
- If not on Collab, either a local instance of jupyterlab, or an IDE.

## 1. Understanding LLMs and Fine-Tuning

This section gives a introductory, high-level overview of large language models and how they work.  

### 1.1 What is a LLM?

In short, a **Large language model (LLM)** is any deep learning model that can comprehend and generate human text $^{[2]}$. In other words, an LLM is a sophisticated artificial intelligence program designed to understand and generate text based on the input it receives. One such example that you may be familiar with is ChatGPT. This is one of many, many language models available for use on the internet. Other notable examples include LLama (Facebook), and Bard (Google). A LLM learns from vast amounts of text data to improve its ability to understand and respond effectively, similarly to a human. 

LLMs are a subset of a wider class of models called **natural language processing models (NLPs)**, computational models designed to understand and interpret human language in order to perform tasks such as text classification, transcription, translation, and more $^{[3]}$. A **Neural Network** is a computational model that works similar to how the human brain functions. Neural networks consist of layers of interconnected nodes, called neurons, that process information (speech, text, images, etc). These networks are trained to learn patterns and relationships in data, making them capable of tasks like image and speech recognition, natural language processing (such as ChatGPT), as well as Generative Adversarial Networks, which generate images from textual prompts $^{[4]}$. 

### 1.2 How does a LLM work?

 In this section, we introduce the basic mechanisms behind large language models powered by generative transformers (GPTs). What makes models such as ChatGPT, Gemini, and LLama so much better than older NLP models is the use of a **transformer architecture** (the "T" in ChatGPT), which allows them to *understand* prompts and generate human-like text. The transformer architecture is a type of neural network that is able to learn context and meaning of a given input text by tracking relationships within the input text $^{[5]}$. 

**1) Vector embedding of input text:** First, the model converts each word in the input sequence into a vector representation known as a token embedding. We won't go into detail as to how this is done; for that, you can consult the [notebook on vector embeddings here.](https://comet.arts.ubc.ca/docs/4_Advanced/advanced_word_embeddings/advanced_word_embeddings_python_version.html)
Additionally, since transformers do not inherently understand the order of tokens, positional encodings are added, which allow the model to understand where each word is relative to other words in the input text. 

**2) Attention Mechanism:** First outlined in the landmark research paper "Attention is all you need"$^{[6]}$ by Google in 2017, the attention mechanism or attention block allows the model to focus in on different parts of the input text and calculate how much *attention* it should pay to every word by comparing it to each other word in the input text. The result is a weighted combination of words' value vectors, reflecting their relevance. This allows the model to prioritize important words and capture meaningful relationships in the sequence, effectively understanding the context and meaning of a text $^{[7]}$ $^{[8]}$. For instance, in the phrase "*The quick brown fox jumps over the lazy...*", the attention mechanism would allow the model to place more emphasis on the words "fox", 'quick" and "brown", and less emphasis on the word "the".

![](transformer.png)

**3) Multi-layer perceptron/feed-forward network:** The multi-layer perceptron, also called a feed-forward network, transforms complex representations of input data by processing it through layers of interconnected "neurons". This transformation helps the network make predictions, classify data, or generate meaningful outputs, using a process called forward propagation. $^{[9]}$. Essentially, it allows the model to map input data to desired outputs effectively. You can think of the feed-forward network as asking a series of questions to the each word in the input sequence $^{[10]}$. For instance, returning to the previous example of *"The quick brown fox jumps over the lazy..."*, the word "fox" could be asked the question "*are you a noun?*" and it's vector embedding would be updated accordingly. 

This process is then repeated a number of times: the resulting vectors are parsed through the attention mechanism, and then back into the feed forward network. Each layer's output becomes the input for the next layer, gradually refining the data. The final layer, which corresponds to the last feed-forward network, produces the network’s prediction. For text generation tasks, this would be take the form of a probability distribution $^{[10]}$.

**4) Unembedding matrix:** The last step multiplies the very last vector in the result of the feed-forward network by a special matrix called the *unembedding matrix*. The result of this multiplication results in a new matrix, for which each entry corresponds to each word in the english language. The values within this vector correspond to the respective probabilities of each word being the correct "next" word $^{[10]}$ $^{[11]}$. 

### 1.3 Weights, Weight Matrices, and Fine-tuning

**Weights:** Weights are parameters within a neural network that are learned during the training process. They determine the strength and direction of the connections in the network $^{[12]}$. Intially, weights are set randomly; during training, the weights are adjusted to minimize the error between the predicted output and the actual output, by minimizing a loss function. This process is known as *gradient descent* $^{[10]}$ $^{[13]}$.

**Weight matrices** are structured collections of weights arranged in matrix form. They represent the connections between layers in a neural network. The operation of passing inputs through the network involves matrix multiplication: the input vector is multiplied by the weight matrix to produce the output vector for the next layer $^{[14]}$.

In the attention mechanism, each word in the input sequence is transformed into three different vectors: the query vector (used to search for relevant information from other words in the sequence), the key vector (represents the words in the sequence and is used to match with query vectors), and the value vector (holds the actual information of the words in the sequence and is used to generate the output of the attention mechanism), using separate weight matrices $^{[14]}$. For example, if the input is a sequence of words represented as vectors, the queries, keys, and values are computed as:

$$Q=W_{Q}(X), K=W_{K}(X), V=W_{V}(X)$$

where $W_{Q}$​, $W_{K}$​, and $W_{V}$​ are weight matrices $^{[14]}$ $^{[15]}$. These vectors are used to calculate attention scores, which determine how much focus each word should give to every other word in the sequence.

![](attention_mechanism.png)


**Fine-tuning** is the process of updating the key, query and value matrices to reflect new data $^{[16]}$. Because the weight matrices contain both the original, general weights and the new adjustments from the fine-tuning process, fine-tuning allows the model to retain the broad, general knowledge from the pre-training phase while specializing in the a new task, such as sentiment analysis, customer feedback, etc. 

### 1.4 Bidirectional VS left-right encoding models

Model LLMs can be grouped into two categories: Those that have bidirectional encoders, and left-right encoders. Left-right encoder models are models that process text sequentially, at any given point in the encoded text sequence, the model can only use information from the current and previous tokens, not future tokens $^{[17]}$. For instance, when processing the text "The quick brown fox jumps over the lazy dog", a left-right encoder processing the word "fox" would only have access to the words "the", "quick" and "brown" when assigning how much attention should be paid to the word "fox".

Bidirectional encoder models, on the other hand, process the input sequence in both directions, from start to end and from end to start. This allows the model to take into account both the left and right context of each token simultaneously $^{[18]}$. This makes bidirectional encoder models particularly strong at sentiment analysis tasks, as they are better able to capture the sentiment assigned to each given word $^{[19]}$. 

**For this reason, if you wish to use large language models for sentiment analysis, it's recommended you use bi-directional encoder models for both greater accuracy and faster training speeds.**

Some popular models include:

- [Finbert](https://huggingface.co/ProsusAI/finbert) (For analyzing financial sentiment)
- [RoBERTa](https://huggingface.co/docs/transformers/en/model_doc/roberta)
- [BERT](https://huggingface.co/google-bert/bert-base-uncased)
- [distilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) 

We will be using BERT for this notebook.

### 1.4 Self tests

#### 1.4.1  Self-test 1

In the phrase "*The quick brown fox jumps over the lazy...*", a left-right encoding model reading the word "fox" would have access to the words _____ when determining the word's ____. 

Assign your answer to an object called `answer_1` as a string in the cell below. For instance, if I were to pick the non-existent option "Z", I would enter `answer_1 = "Z"`. 

- A) "jumps", "over", "the", and "lazy". Relevance.
- B) "The", "quick", "brown", "jumps", "over", "the", and "lazy". Vector embedding.
- C) "The", "quick", and "brown", Vector embedding.
- D) "jumps", "over", "the", and "lazy". Vector embedding.
- E) "The", "quick", and "brown", Relevance.
- F) "The", "quick", "brown", "jumps", "over", "the", and "lazy". Relevance.


In [2]:
#input your answer here

In [3]:
import hashlib
from hashlib import sha256

h=hashlib.new("SHA256")
h.update(answer_1.encode())
if str(h.hexdigest()) == "a9f51566bd6705f7ea6ad54bb9deb449f795582d6529a0e22207b8981233ec58":
    print("correct! \U0001f600")
else: print("incorrect, recall the the difference between left-right and bidirectional encoder models.")

NameError: name 'answer_1' is not defined

In [ ]:
correct = "C"
h=hashlib.new("SHA256")
h.update(correct.encode())
h.hexdigest()

#### 1.4.2 Self Test 2

Suppose an LLM was given the following text and tasked to perform sentiment analysis: "It's a beautiful sunny day outside". It's first take would be to **embed** each word. Which of the following is a reasonable embedding for the word "sunny"?

- A) isjfk29ndlsavbm4_2u3n
- B) " sun-ny "
- C) <3820.2, 38573.6, 1826.2, 23.3, ... 4958.3>
- D) 🌞

Assign your answer to an object called `answer_2` as a string in the cell below. For instance, if I were to pick the non-existent option "Z", I would enter `answer_2 = "Z"`. 

In [ ]:
#input your answer here

In [ ]:
import hashlib
from hashlib import sha256

h=hashlib.new("SHA256")
h.update(answer_2.encode())
if str(h.hexdigest()) == '6b23c0d5f35d1b11f9b683f0b0a617355deb11277d91ae091d399c655b87940d':
    print("correct! \U0001f600")
else: print("incorrect, recall that embeddings have both magnitude and direction.")

## 2. Setting up
Before we begin, we'll need to create a new python environment for our required libraries, as well as install CUDA. 

### 2.1 Creating an envrionment

**Skip this step if you are using Google Collab.**

Let's first create a python environment, using conda.

1) Make sure you have miniconda installed, and open up the miniconda prompt.

2) In the miniconda prompt, enter `conda create -n llm_finetuning jupyter`. This will create a new environment called llm_finetuning, with jupyter installed.

3) Next, activate the environment by typing `conda activate llm_finetuning`. 

### 2.2 Installing required libraries

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate peft optimum bitsandbytes
!pip install -q huggingface_hub
!pip install -q -U sentence-transformers
!pip install -q bertopic umap-learn
!pip install -q kaleido wordcloud protobuf
!pip install -q spacy
!python -m spacy download en_core_web_sm
!pip uninstall -y hdbscan
!pip install -q git+https://github.com/scikit-learn-contrib/hdbscan@master#egg=hdbscan
!pip install -q nltk
!pip install -q arxiv

ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 20.7 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Found existing installation: hdbscan 0.8.40
Uninstalling hdbscan-0.8.40:
  Successfully uninstalled hdbscan-0.8.40


In [4]:
import os
import sys
import json
import random
import datetime
import dill
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForMaskedLM
from datasets import load_dataset
from collections import defaultdict
import spacy
from tqdm import tqdm # CHANGED: IMPORT FROM TQDM.NOTEBOOK FOR BETTER UI
import logging

# BERTopic & sentence-transformers
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer, models
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
import arxiv
from torch.optim import AdamW

/opt/miniconda3/envs/new_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Finetuning a BERT model on economic paper abstracts

Our purpose for applying finetuning here is simple. BERT is a generalized model, trained on the entirety of the English Wikipedia and Brown corpus. If we want to infer specific patterns about our corpus of abstracts, we need to teach it to *speak economist*. We do so by finetuning it on thousands of economic abstracts, getting it to learn the complex statistical relationships between economic terms. 



In [5]:
SEED1 = 6548
SEED2 = 3091
MAX_LENGTH = 128
BASE_DIR = "output/bert"
VERSION = "1.2"

random.seed(SEED1)
np.random.seed(SEED2)
torch.manual_seed(SEED1)

In [6]:
def set_up_logger(log_dir):
    """
    Creates logging directory + file for any issues/warnings that show up while executing code.
    Can be found at BASE_DIR/train.log
    """
    os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger("train_logger")
    logger.setLevel(logging.DEBUG)
    fh = logging.FileHandler(os.path.join(log_dir, "train.log"), mode='w') 
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    logger.addHandler(fh)
    logger.addHandler(logging.StreamHandler(sys.stdout))
    return logger

logger = set_up_logger(BASE_DIR)

As a simple example, we'll be using a collection of about 2000 abstracts pulled from Arxiv's *General Economics* category, which includes "general methodological, applied, and empirical contributions to economics." Our first step is to load the dataset. 

In [ ]:
# loading data
logger.info("Fetching data from ArXiv API")

# Search for General Economics (econ.GN) papers
# We fetch 2000 results sorted by relevance
client = arxiv.Client()
search = arxiv.Search(
    query = "cat:econ.GN",
    max_results = 20000,
    sort_by = arxiv.SortCriterion.Relevance)

train_data = [] 
validation_data = []
test_data = []

# Collect results
all_papers = []
results = client.results(search)

all_timestamps = []

logger.info("Downloading abstracts...")
for r in tqdm(results, total=20000, desc="Fetching Papers"):
    abstract = r.summary.replace("\n", " ")
    
    if len(abstract) > 50: 
        entry = {"Abstract": abstract}
        all_papers.append(entry)
        
        all_timestamps.append(r.published)

# ... existing split logic ...

# Split the data
total_docs = len(all_papers)
split_1 = int(total_docs * 0.8)
split_2 = int(total_docs * 0.9)
train_timestamps = all_timestamps[:split_1]

train_data = all_papers[:split_1]
validation_data = all_papers[split_1:split_2]
test_data = all_papers[split_2:]

logger.info(f"Data Loaded - Train: {len(train_data)}, Val: {len(validation_data)}, Test: {len(test_data)}")



Fetching data from ArXiv API


Fetching Papers:  32%|███▏      | 6413/20000 [04:00<08:29, 26.65it/s]

Data Loaded - Train: 5129, Val: 641, Test: 642


Now we are ready to prepare the dataset. 
# 3.1 Preprocessing 
We want to convert the text in the abstracts to 'tokens' a model can understand. To do this, we will be using a RoBERTa tokenizer and splitting the dataset into training, validation, and testing datatsets.

In [22]:
class EconlitDataset(Dataset):
    """
    Parameters:
        - dataset: list of dictionaries
        - max_length: maximum number of tokens in any given abstract
        - tokenizer: BERT tokenizer object
        - mlm_probability: percent of tokens that will be masked in a given abstract

    Functions:
        - __len__: length of dataset
        - __getitem__: given an index in list of dictionaries, pulls dictionary at index idx,
        tokenizes abstract, mask self.mlm_probability tokens of said abstract
    """
    def __init__(self, dataset, max_length, tokenizer, mlm_probability=0.15):
        self.dataset = dataset
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        text = str(item.get("Abstract", ""))

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze(0) # converts tensor shape from [1, MAX_LENGTH] to [MAX_LENGTH]
        # array of size 1 x 128
        attention_mask = encoding["attention_mask"].squeeze(0) # boolean mask for which tokens are padding and which are not

        labels = input_ids.clone()
        probability_matrix = torch.full(labels.shape, self.mlm_probability)

        special_tokens_mask = self.tokenizer.get_special_tokens_mask(
            input_ids.tolist(), already_has_special_tokens=True
        ) # to prevent special tokens from being masked

        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0) # to prevent special tokens from being masked

        masked_indices = torch.bernoulli(probability_matrix).bool() #boolean tensor of size BATCH_SIZE which determines which tokens are to be masked
        labels[~masked_indices] = -100 

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# CHANGED: USING THE LISTS WE CREATED FROM ARXIV INSTEAD OF LOADING FILES AGAIN
training_dataset = EconlitDataset(train_data, MAX_LENGTH, tokenizer)
validation_dataset = EconlitDataset(validation_data, MAX_LENGTH, tokenizer)
testing_dataset = EconlitDataset(test_data, MAX_LENGTH, tokenizer)

train_loader = DataLoader(training_dataset, batch_size=16, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=16)

logger.info("Tokenized Datasets Created:")
logger.info(f"Training samples: {len(training_dataset)}")
logger.info(f"Validation samples: {len(validation_dataset)}")
logger.info(f"Testing samples: {len(testing_dataset)}")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 036affea-7164-460d-aa79-662547c84a66)')' thrown while requesting HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Tokenized Datasets Created:
Training samples: 1600
Validation samples: 200
Testing samples: 200


In [28]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = RobertaForMaskedLM.from_pretrained("roberta-base")
model.to(device)

logger.info(f"Model loaded and moved to device: {device}")

# optimizing stuff
LEARNING_RATE = 2e-5
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# training
def train_mlm(model, data_loader, optimizer, device, epoch):
    model.train()
    total_loss = 0

    for step, batch in enumerate(tqdm(data_loader, desc=f"Epoch {epoch}")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels) # forward pass
        loss = outputs.loss
        loss.backward() # backpropagation
        optimizer.step() # update parameters

        total_loss += loss.item()

        if step % 100 == 0 and step != 0:
            logger.info(f"Step {step} - Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(data_loader)
    logger.info(f"Epoch {epoch} finished with average loss: {avg_loss:.4f}")
    return avg_loss

# epochs
EPOCHS = 1 # CHANGED: REDUCED TO 1 FOR DEMO SPEED

for epoch in range(EPOCHS):
    logger.info(f"Starting epoch {epoch + 1}/{EPOCHS}")
    train_mlm(model, train_loader, optimizer, device, epoch + 1)


Model loaded and moved to device: cpu
Starting epoch 1/1


Epoch 1: 100%|██████████| 100/100 [04:13<00:00,  2.53s/it]

Epoch 1 finished with average loss: 0.0029


In [29]:
# saving model
save_dir = f"{BASE_DIR}/models/roberta_mlm_v{VERSION}"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

logger.info(f"Saved model and tokenizer to {save_dir}")

# roberta 
model_path = f"{BASE_DIR}/models/roberta_mlm_v{VERSION}"

word_embedding_model = models.Transformer(model_path, max_seq_length=512)
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
custom_sentence_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])


Saved model and tokenizer to output/bert/models/roberta_mlm_v1.2


Some weights of RobertaModel were not initialized from the model checkpoint at output/bert/models/roberta_mlm_v1.2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
import nltk

In [12]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ireneberezin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ireneberezin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ireneberezin/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ireneberezin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [36]:
docs = [entry['Abstract'] for entry in train_data]
print(f"Loaded {len(docs)} abstracts for Topic Modeling.")

import nltk
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 
import string

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text: str, lemmatize = True, stem = True) -> str:
    """
    Takes in a complete abstract as a string and preprocesses it.
    """
    tokens = word_tokenize(text.lower())
    cleaned_words = []
    for word in tokens:
        if word.isalpha() and word not in stop_words:
            if lemmatize and stem:
                lemmatized_word = lemmatizer.lemmatize(word)
                stemmed_word = stemmer.stem(lemmatized_word)
                cleaned_words.append(stemmed_word)
            elif lemmatize and not stem:
                lemmatized_word = lemmatizer.lemmatize(word)
                cleaned_words.append(lemmatized_word)
            elif not lemmatize and stem:
                stemmed_word = stemmer.stem(word)
                cleaned_words.append(stemmed_word)
            else: 
                cleaned_words.append(word)
        else: pass
    return " ".join(cleaned_words)

docs_cleaned = [preprocess_text(doc) for doc in docs]

# bertopic
topic_model = BERTopic(embedding_model=custom_sentence_model, verbose=True)
topics, probs = topic_model.fit_transform(docs_cleaned)

# visualization of topics
fig = topic_model.visualize_topics()
fig.show()

Loaded 5129 abstracts for Topic Modeling.


2025-11-24 15:11:05,695 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 161/161 [01:14<00:00,  2.15it/s]
2025-11-24 15:12:20,788 - BERTopic - Embedding - Completed ✓
2025-11-24 15:12:20,789 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-24 15:12:22,138 - BERTopic - Dimensionality - Completed ✓
2025-11-24 15:12:22,140 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-24 15:12:22,216 - BERTopic - Cluster - Completed ✓
2025-11-24 15:12:22,220 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-24 15:12:22,351 - BERTopic - Representation - Completed ✓


In [35]:
# Visualize the top keywords for the top 10 topics
fig_bar = topic_model.visualize_barchart(top_n_topics=5)
fig_bar.show()